# Activité Pratique N°5 — Partie A : LangChain


## Configuration


In [1]:
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI

load_dotenv(override=True)
GROQ_API_KEY = os.getenv("GROQ_API_KEY")
GROQ_BASE_URL = "https://api.groq.com/openai/v1"
MODEL_NAME = "openai/gpt-oss-20b"

assert GROQ_API_KEY, "Ajoutez GROQ_API_KEY dans partie5/agentic-mcp/.env"
llm = ChatOpenAI(model=MODEL_NAME, api_key=GROQ_API_KEY, base_url=GROQ_BASE_URL, temperature=0)
print(f"Configuration OK | Modèle Groq : {MODEL_NAME}")

Configuration OK | Modèle Groq : openai/gpt-oss-20b


## 1. Simple Agent


In [2]:
from langchain.agents import create_agent

simple_agent = create_agent(
    model=llm,
    tools=[],
    system_prompt="Tu es un assistant pédagogique concis en français.",
)

res = simple_agent.invoke({
    "messages": [{"role": "user", "content": "Explique en 2 phrases ce qu'est un agent IA."}]
})
print(res["messages"][-1].content)

Un agent IA est un logiciel autonome capable d’observer son environnement, de traiter ces informations et d’agir pour atteindre un objectif prédéfini. Il combine perception, raisonnement et action, souvent via des algorithmes d’apprentissage, pour s’adapter et optimiser ses performances.


## 2. Mémoire


In [3]:
from langgraph.checkpoint.memory import InMemorySaver

agent_memoire = create_agent(
    model=llm,
    tools=[],
    checkpointer=InMemorySaver(),
)

config_mem = {"configurable": {"thread_id": "etudiant-1"}}

r1 = agent_memoire.invoke(
    {"messages": [{"role": "user", "content": "Bonjour, je m'appelle Mohamed et j'étudie l'IA générative."}]},
    config=config_mem,
)
print("Tour 1 :", r1["messages"][-1].content)

r2 = agent_memoire.invoke(
    {"messages": [{"role": "user", "content": "Comment je m'appelle et qu'est-ce que j'étudie ?"}]},
    config=config_mem,
)
print("Tour 2 :", r2["messages"][-1].content)

Tour 1 : Bonjour Mohamed ! Ravi de faire votre connaissance.  
Vous étudiez l’IA générative ? C’est un domaine passionnant !  
Sur quel type de projets ou de technologies vous concentrez‑vous ? (ex. modèles de texte, d’image, de musique, etc.) ou avez‑vous des questions spécifiques auxquelles je pourrais répondre ?
Tour 2 : Vous vous appelez **Mohamed** et vous étudiez **l’IA générative**.


## 3. Tools personnalisés et prédéfinis (DuckDuckGo, Tavily, PythonREPL)


In [4]:
from langchain.tools import tool
from langchain_experimental.tools import PythonREPLTool
from ddgs import DDGS

@tool
def calculer_ttc(montant_ht: float, tva: float = 0.20) -> str:
    """Calcule le montant TTC à partir d'un montant HT et d'un taux de TVA."""
    return f"{montant_ht * (1 + tva):.2f} MAD"

@tool
def recherche_duckduckgo(requete: str) -> str:
    """Recherche des informations récentes sur le web avec DuckDuckGo."""
    results = DDGS().text(query=requete, max_results=3)
    if not results:
        return "Aucun résultat trouvé."
    return "\n".join(f"- {r['title']}: {r['body']}" for r in results)

python_repl = PythonREPLTool()
outils = [calculer_ttc, recherche_duckduckgo, python_repl]

if os.getenv("TAVILY_API_KEY"):
    from langchain_tavily import TavilySearch
    outils.append(TavilySearch(max_results=3))
    print("TavilySearch activé.")
else:
    print("TAVILY_API_KEY absent : exécution sans TavilySearch.")

agent_tools = create_agent(model=llm, tools=outils)

res_tools = agent_tools.invoke({
    "messages": [{"role": "user", "content": "Calcule le TTC pour 500 MAD HT avec 20% de TVA, puis calcule 15 * 24 en Python."}]
})
print(res_tools["messages"][-1].content)

TAVILY_API_KEY absent : exécution sans TavilySearch.
- **TTC** pour 500 MAD HT avec 20 % de TVA : **600 MAD**  
- **Résultat de 15 × 24** en Python : **360**


<ipython-input-1-add844171add>:2: DeprecationWarning: `langchain-experimental` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-experimental/issues/87 for details.
  from langchain_experimental.tools import PythonREPLTool
Python REPL can execute arbitrary code. Use with caution.


## 4. Middlewares (dynamic_model, dynamic_prompt, tool_error_handling, Guardrails, HITL)


In [5]:
from typing import TypedDict, Any
from langchain.agents.middleware import (
    wrap_model_call,
    dynamic_prompt,
    wrap_tool_call,
    PIIMiddleware,
    HumanInTheLoopMiddleware,
    after_agent,
    AgentState,
    ModelRequest,
)
from langchain.messages import ToolMessage, AIMessage
from langgraph.runtime import Runtime
from langgraph.types import Command

# 1. Dynamic model
fast_llm = ChatOpenAI(model="openai/gpt-oss-20b", api_key=GROQ_API_KEY, base_url=GROQ_BASE_URL, temperature=0)
smart_llm = ChatOpenAI(model="openai/gpt-oss-120b", api_key=GROQ_API_KEY, base_url=GROQ_BASE_URL, temperature=0)

@wrap_model_call
def dynamic_model(request: ModelRequest, handler):
    model = smart_llm if len(request.state["messages"]) > 4 else fast_llm
    return handler(request.override(model=model))

# 2. Dynamic prompt
class Context(TypedDict):
    user_role: str

@dynamic_prompt
def dynamic_role_prompt(request: ModelRequest) -> str:
    role = (request.runtime.context or {}).get("user_role", "student")
    if role == "expert":
        return "Tu es un assistant technique détaillé."
    return "Tu es un assistant simple et pédagogique."

# 3. Tool error handling
@wrap_tool_call
def handle_tool_errors(request, handler):
    try:
        return handler(request)
    except Exception as e:
        return ToolMessage(
            content=f"Erreur outil interceptée : {e}",
            tool_call_id=request.tool_call["id"],
        )

@tool
def diviser(a: float, b: float) -> float:
    """Divise a par b."""
    return a / b

# 4. Guardrail personnalisé après réponse
@after_agent
def safety_guardrail(state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
    if state["messages"] and isinstance(state["messages"][-1], AIMessage):
        txt = state["messages"][-1].content.lower()
        if "mot de passe" in txt:
            state["messages"][-1].content = "Réponse bloquée par le garde-fou."
    return None

agent_mw = create_agent(
    model=llm,
    tools=[diviser],
    context_schema=Context,
    middleware=[
        dynamic_model,
        dynamic_role_prompt,
        handle_tool_errors,
        PIIMiddleware("email", strategy="redact", apply_to_input=True),
        safety_guardrail,
    ],
)

res_mw = agent_mw.invoke(
    {"messages": [{"role": "user", "content": "Mon email est test@example.com. Appelle l'outil diviser avec a=10 et b=0."}]},
    context={"user_role": "student"},
)
print("Middleware test :", res_mw["messages"][-1].content)

# 5. Human-in-the-Loop
@tool
def envoyer_email(destinataire: str, sujet: str) -> str:
    """Envoie un email."""
    return f"Email envoyé à {destinataire} ({sujet})."

agent_hitl = create_agent(
    model=llm,
    tools=[envoyer_email],
    middleware=[HumanInTheLoopMiddleware(interrupt_on={"envoyer_email": True})],
    checkpointer=InMemorySaver(),
)

cfg_hitl = {"configurable": {"thread_id": "hitl-1"}}
pause = agent_hitl.invoke(
    {"messages": [{"role": "user", "content": "Envoie un email à prof@emsi.ma avec le sujet TP5."}]},
    config=cfg_hitl,
)
print("Interruption HITL :", "__interrupt__" in pause)

reprise = agent_hitl.invoke(
    Command(resume={"decisions": [{"type": "approve"}]}),
    config=cfg_hitl,
)
print("Après approbation :", reprise["messages"][-1].content)

Middleware test : Je suis désolé, mais il n’est pas possible de diviser par zéro.  
Si vous avez besoin d’un autre calcul ou d’une autre valeur pour b, n’hésitez pas à me le dire !
Interruption HITL : True
Après approbation : Email envoyé à prof@emsi.ma (TP5).


## 5. Chatbot RAG Agentique


In [6]:
from langchain_core.documents import Document
from langchain_core.tools import create_retriever_tool
from langchain_community.embeddings import FastEmbedEmbeddings
from langchain_community.vectorstores import FAISS

docs = [
    Document(page_content="Le module IA Générative à l'EMSI couvre les Transformers, LangChain, LangGraph et le protocole MCP."),
    Document(page_content="LangChain permet de créer des agents ReAct avec des tools, de la mémoire et des middlewares."),
    Document(page_content="Un RAG agentique donne à l'agent un outil de recherche vectorielle pour décider quand interroger la base documentaire."),
]

embeddings = FastEmbedEmbeddings()
vectorstore = FAISS.from_documents(docs, embeddings)
retriever_tool = create_retriever_tool(
    retriever=vectorstore.as_retriever(search_kwargs={"k": 2}),
    name="recherche_cours",
    description="Recherche des informations sur le cours IA Générative.",
)

chatbot_rag = create_agent(
    model=llm,
    tools=[retriever_tool],
    system_prompt="Utilise l'outil recherche_cours pour répondre aux questions sur le cours.",
    checkpointer=InMemorySaver(),
)

cfg_rag = {"configurable": {"thread_id": "rag-chat-1"}}
for q in [
    "Que couvre le module IA Générative à l'EMSI ?",
    "Et quel est le rôle du RAG agentique ?",
]:
    ans = chatbot_rag.invoke({"messages": [{"role": "user", "content": q}]}, config=cfg_rag)
    print(f"Q: {q}\nR: {ans['messages'][-1].content}\n")

Q: Que couvre le module IA Générative à l'EMSI ?
R: Le module **IA Générative** de l’EMSI aborde les sujets suivants :

| Thème | Contenu clé |
|-------|-------------|
| **Transformers** | Architecture, entraînement, fine‑tuning, applications (NLP, vision, multimodal) |
| **LangChain** | Construction d’applications IA avec des chaînes de prompts, intégration de modèles, gestion d’état |
| **LangGraph** | Modélisation de flux de travail et de dialogues, graphes de décision pour agents |
| **Protocole MCP** | Méthodologie de mise en production et de monitoring des modèles génératifs |
| **RAG (Retrieval‑Augmented Generation)** | Agents qui utilisent une recherche vectorielle pour interroger une base documentaire au besoin |

En résumé, le module couvre l’architecture des modèles de pointe, les outils de développement (LangChain, LangGraph), les bonnes pratiques de production (MCP) et les techniques d’augmentation de génération par recherche (RAG).

Q: Et quel est le rôle du RAG agentique

<ipython-input-1-d852a24f2558>:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.embeddings import FastEmbedEmbeddings
Fetching 5 files: 100%|##########| 5/5 [00:03<00:00,  1.36it/s]
